In [3]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from openai import OpenAI

# 1. Load environment variables from your .env file
# (Note: standard filename is ".env", not ".evn" — adjust the path if your file is elsewhere)
env_path = "/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/.env"
loaded = load_dotenv(dotenv_path=env_path, override=True)

# Optional fallback if running from a subdirectory and not using the hardcoded path above:
# if not loaded:
#     load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)

# 2. Verify OPENAI_API_KEY was loaded
assert os.environ.get("OPENAI_API_KEY"), (
    f"OPENAI_API_KEY not found. Check that it is set in {env_path}"
)

# The OpenAI client reads OPENAI_API_KEY from the environment automatically
client = OpenAI()

# Use a valid OpenAI model identifier (e.g., "gpt-4o-mini", "gpt-4.1-mini", or whichever model your account supports)
MODEL = "gpt-4o-mini"
print(f"Client ready. Using model: {MODEL}")

Client ready. Using model: gpt-4o-mini


In [8]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two whole numbers together and return the product."""
    return a * b


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city name."""
    return f"Sunny and 26°C in {city}"


# Tell the system prompt to briefly state WHICH tool it chose and WHY before calling it
agent = create_agent(
   
   model=f"openai:{MODEL}",
    tools=[multiply_numbers],
    system_prompt=(
        "You are a helpful assistant. "
        "Whenever you call a tool, first write a 1-sentence explanation of your "
        "tool selection logic (why this tool matches the request and why other tools do not), "
        "and then invoke the tool."
    ),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is 7 times 8?"}]}
)


# 6. Print the final response
print(result["messages"][-1].content)

I will use the `multiply_numbers` function to calculate the product of 7 and 8, as it is specific for multiplying two whole numbers. Other tools are not suitable for this arithmetic operation. 

Now, I will perform the multiplication. 

Invoking the tool: 

```json
{"a":7,"b":8}
```


In [ ]:
def show_selection_logic(agent_result):
    available_tools = [t.name for t in [multiply_numbers, get_weather]]
    print(f"Available tools given to model: {available_tools}\n")

    for step, msg in enumerate(agent_result["messages"], start=1):
        msg_type = getattr(msg, "type", type(msg).__name__)

        if msg_type == "ai":
            # 1. Model's written selection logic / reasoning
            if msg.content:
                print(f"[Step {step} - Model Selection Logic]:\n  {msg.content}")

            # 2. The actual tool call(s) emitted
            for call in getattr(msg, "tool_calls", None) or []:
                print(
                    f"[Step {step} - Tool Invoked]:\n"
                    f"  Tool: {call['name']} | Args: {call['args']}"
                )

        elif msg_type == "tool":
            print(
                f"[Step {step} - Tool Result ({getattr(msg, 'name', '')})]:\n"
                f"  {msg.content}\n"
            )

        elif msg_type == "ai" and not getattr(msg, "tool_calls", None):
            print(f"[Step {step} - Final Answer]:\n  {msg.content}")


show_selection_logic(result)